In [4]:

"""
Project 2 - Library Book Inventory Manager
- Add books, search by title/author, issue/return
- Book and Library classes
- Dictionary for fast book lookup
- Saves and loads data using a JSON file
- Reports total, issued, and available books
"""

import json
import os

DATA_FILE = "library_data.json"


class Book:
    def __init__(self, book_id, title, author, issued=False):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.issued = issued

    def to_dict(self):
        return {
            "book_id": self.book_id,
            "title": self.title,
            "author": self.author,
            "issued": self.issued
        }

    @staticmethod
    def from_dict(data):
        return Book(
            data["book_id"],
            data["title"],
            data["author"],
            data.get("issued", False)
        )

    def __str__(self):
        status = "Issued" if self.issued else "Available"
        return (
            f"{self.book_id:<10}"
            f"{self.title:<35}"
            f"{self.author:<28}"
            f"{status}"
        )


class Library:
    def __init__(self, filename=DATA_FILE):
        self.filename = filename
        self.books = {}
        self.load()

        # Add sample books only if the data file does not exist.
        if not os.path.exists(self.filename):
            self.add_sample_books()

    def load(self):
        if not os.path.exists(self.filename):
            return

        try:
            with open(self.filename, "r", encoding="utf-8") as file:
                data = json.load(file)

            if not isinstance(data, list):
                raise ValueError("Invalid data format")

            self.books = {
                item["book_id"]: Book.from_dict(item)
                for item in data
            }

        except (json.JSONDecodeError, KeyError, TypeError, ValueError):
            print("Warning: Invalid data file. Starting with an empty library.")
            self.books = {}

    def save(self):
        try:
            with open(self.filename, "w", encoding="utf-8") as file:
                json.dump(
                    [book.to_dict() for book in self.books.values()],
                    file,
                    indent=4
                )
        except OSError as error:
            print(f"Error saving library data: {error}")

    def add_sample_books(self):
        sample_books = [
            ("CS101", "Introduction to Algorithms",
             "Thomas H. Cormen"),
            ("CS102", "Operating System Concepts",
             "Abraham Silberschatz"),
            ("CS103", "Computer Networking",
             "James Kurose"),
            ("CS104", "Database System Concepts",
             "Abraham Silberschatz"),
            ("CS105", "The C Programming Language",
             "Brian Kernighan")
        ]

        for book_id, title, author in sample_books:
            self.books[book_id] = Book(book_id, title, author)

        self.save()

    def add_book(self, book_id, title, author):
        book_id = book_id.strip()
        title = title.strip()
        author = author.strip()

        if not book_id or not title or not author:
            print("Error: All fields are required.")
            return

        if book_id in self.books:
            print("Error: A book with this ID already exists.")
            return

        self.books[book_id] = Book(book_id, title, author)
        self.save()
        print(f"Book '{title}' added successfully.")

    def search(self, keyword, field):
        keyword = keyword.strip().casefold()

        if not keyword:
            print("Please enter a search keyword.")
            return []

        return [
            book for book in self.books.values()
            if keyword in getattr(book, field).casefold()
        ]

    def issue_book(self, book_id):
        book = self.books.get(book_id.strip())

        if book is None:
            print("Error: Book not found.")
        elif book.issued:
            print("Error: This book is already issued.")
        else:
            book.issued = True
            self.save()
            print(f"Book '{book.title}' issued successfully.")

    def return_book(self, book_id):
        book = self.books.get(book_id.strip())

        if book is None:
            print("Error: Book not found.")
        elif not book.issued:
            print("Error: This book has not been issued.")
        else:
            book.issued = False
            self.save()
            print(f"Book '{book.title}' returned successfully.")

    @staticmethod
    def print_books(books):
        if not books:
            print("No books found.")
            return

        print(
            f"{'ID':<10}"
            f"{'Title':<35}"
            f"{'Author':<28}"
            f"Status"
        )
        print("-" * 90)

        for book in books:
            print(book)

    def list_books(self):
        self.print_books(list(self.books.values()))

    def report(self):
        total = len(self.books)
        issued = sum(1 for book in self.books.values() if book.issued)
        available = total - issued

        print("\n========== LIBRARY REPORT ==========")
        print(f"Total books     : {total}")
        print(f"Issued books    : {issued}")
        print(f"Available books : {available}")
        print("====================================")


def menu():
    print("\n===== Library Book Inventory Manager =====")
    print("1. Add Book")
    print("2. List All Books")
    print("3. Search by Title")
    print("4. Search by Author")
    print("5. Issue Book")
    print("6. Return Book")
    print("7. Report")
    print("8. Exit")


def main():
    library = Library()

    while True:
        menu()
        choice = input("Enter choice (1-8): ").strip()

        if choice == "1":
            print("\n--- Add New Book ---")
            book_id = input("Book ID: ")
            title = input("Title: ")
            author = input("Author: ")
            library.add_book(book_id, title, author)

        elif choice == "2":
            print("\n--- All Library Books ---")
            library.list_books()

        elif choice == "3":
            keyword = input("Enter title keyword: ")
            results = library.search(keyword, "title")
            library.print_books(results)

        elif choice == "4":
            keyword = input("Enter author name or keyword: ")
            results = library.search(keyword, "author")
            library.print_books(results)

        elif choice == "5":
            book_id = input("Enter Book ID to issue: ").strip()
            library.issue_book(book_id)

        elif choice == "6":
            book_id = input("Enter Book ID to return: ").strip()
            library.return_book(book_id)

        elif choice == "7":
            library.report()

        elif choice == "8":
            print("Thank you for using the Library Book Inventory Manager!")
            break

        else:
            print("Invalid choice. Please enter a number from 1 to 8.")


if __name__ == "__main__":
    main()



===== Library Book Inventory Manager =====
1. Add Book
2. List All Books
3. Search by Title
4. Search by Author
5. Issue Book
6. Return Book
7. Report
8. Exit
Enter choice (1-8): 1

--- Add New Book ---
Book ID: CS107
Title: Data Structures and Algorithms in Python
Author:  Michael T. Goodrich
Book 'Data Structures and Algorithms in Python' added successfully.

===== Library Book Inventory Manager =====
1. Add Book
2. List All Books
3. Search by Title
4. Search by Author
5. Issue Book
6. Return Book
7. Report
8. Exit
Enter choice (1-8): 2

--- All Library Books ---
ID        Title                              Author                      Status
------------------------------------------------------------------------------------------
CS101     Introduction to Algorithms         Thomas H. Cormen            Available
CS102     Operating System Concepts          Abraham Silberschatz        Available
CS103     Computer Networking                James Kurose                Available
CS104  